# Patient Risk Modelling — Random Forest vs XGBoost vs LightGBM

Trains and compares the three tree ensembles on two datasets:

| Dataset | Rows | Task | Source |
|---|---|---|---|
| Maternal Health Risk | 1014 → **452 after dedup** | 3-class risk | auto-downloaded from UCI |
| Patient Monitoring | 60,000 | 5-class condition | **you upload this** |

Also flags the clinical issues each patient may be facing.

**Runtime:** CPU is fine (~5 min total). No GPU needed.

> Educational project on public research datasets. Not validated for clinical use.

---
## Step 0 — Install dependencies

In [ ]:
!pip install -q ucimlrepo xgboost lightgbm

import xgboost, lightgbm, sklearn, pandas as pd, numpy as np
print("xgboost ", xgboost.__version__)
print("lightgbm", lightgbm.__version__)
print("sklearn ", sklearn.__version__)
print("pandas  ", pd.__version__)

## Step 1 — Imports

In [ ]:
import warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, roc_auc_score)
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
plt.rcParams["figure.dpi"] = 110

## Step 2 — Get the maternal dataset

Pulled straight from the UCI ML Repository (dataset #863) — no API key needed.
This is the canonical version of the three identical CSVs you had locally.

In [ ]:
from ucimlrepo import fetch_ucirepo

uci = fetch_ucirepo(id=863)
maternal_raw = pd.concat([uci.data.features, uci.data.targets], axis=1)
maternal_raw = maternal_raw.rename(columns={"BS": "BloodSugar", "Blood sugar": "BloodSugar"})

print("Shape:", maternal_raw.shape)
print()
print(maternal_raw.head())
print()
print(maternal_raw["RiskLevel"].value_counts())

## Step 3 — Upload the monitoring dataset

This one isn't public, so upload `Synthetic_patient-HealthCare-Monitoring_dataset.csv`
when the file picker appears.

*Alternative:* put the file in your Drive and swap the cell for
`from google.colab import drive; drive.mount('/content/drive')`, then
`pd.read_csv('/content/drive/MyDrive/<path>.csv')`.

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
fname = next(iter(uploaded))
monitoring_raw = pd.read_csv(io.BytesIO(uploaded[fname]))

monitoring_raw = monitoring_raw.rename(columns={
    "Patient Number": "PatientID",
    "Heart Rate (bpm)": "HeartRate",
    "SpO2 Level (%)": "SpO2",
    "Systolic Blood Pressure (mmHg)": "SystolicBP",
    "Diastolic Blood Pressure (mmHg)": "DiastolicBP",
    "Body Temperature (\u00b0C)": "BodyTemp",
    "Fall Detection": "FallDetected",
    "Predicted Disease": "Condition",
    "Data Accuracy (%)": "DataAccuracy",
})

print("Shape:", monitoring_raw.shape)
print(monitoring_raw["Condition"].value_counts())

## Step 4 — Clean the data (this is where the real result is)

Two things matter here:

1. **562 of 1014 maternal rows are exact duplicates.** Dropping them *before*
   the split is non-negotiable — otherwise identical rows land in both train and
   test and the model gets graded on rows it memorised. The cell below quantifies
   exactly how much that inflates your score.
2. **The four `*Alert` columns in the monitoring set are deterministic threshold
   functions of the vitals.** Verified at 100% match. They're dropped — keeping
   them leaks the labelling rule.

In [ ]:
def clean_maternal(df, drop_duplicates=True):
    d = df.copy()
    n_raw = len(d)

    # 7 bpm heart rates are impossible sensor errors -> median impute
    n_bad_hr = int((d["HeartRate"] < 40).sum())
    d.loc[d["HeartRate"] < 40, "HeartRate"] = np.nan
    d["HeartRate"] = d["HeartRate"].fillna(d["HeartRate"].median())

    n_dupes = int(d.duplicated().sum())
    if drop_duplicates:
        d = d.drop_duplicates().reset_index(drop=True)

    d["RiskLevel"] = d["RiskLevel"].astype(str).str.strip()
    print(f"raw rows           : {n_raw}")
    print(f"impossible HR rows : {n_bad_hr}")
    print(f"duplicate rows     : {n_dupes}  ({n_dupes/n_raw:.0%})")
    print(f"rows kept          : {len(d)}")
    return d


def clean_monitoring(df):
    d = df.copy()
    alerts = ["Heart Rate Alert", "SpO2 Level Alert",
              "Blood Pressure Alert", "Temperature Alert"]
    checks = {
        "Heart Rate Alert": np.where((d.HeartRate < 60) | (d.HeartRate > 100), "ABNORMAL", "NORMAL"),
        "SpO2 Level Alert": np.where(d.SpO2 < 95, "ABNORMAL", "NORMAL"),
        "Blood Pressure Alert": np.where((d.SystolicBP > 140) | (d.DiastolicBP > 90), "ABNORMAL", "NORMAL"),
        "Temperature Alert": np.where(d.BodyTemp > 37.5, "ABNORMAL", "NORMAL"),
    }
    for col, rule in checks.items():
        if col in d.columns:
            print(f"{col:22s} reproduced by threshold rule: {(rule == d[col]).mean():.1%}")

    d = d.drop(columns=[c for c in alerts if c in d.columns])
    print(f"\nduplicates: {d.duplicated().sum()}   missing: {d.isna().sum().sum()}   rows: {len(d)}")
    return d


print("--- MATERNAL ---")
maternal = clean_maternal(maternal_raw)
print("\n--- MONITORING ---")
monitoring = clean_monitoring(monitoring_raw)

## Step 5 — Clinically-motivated features

Pulse pressure, mean arterial pressure and shock index are standard bedside
derivations. Tree models can technically learn them, but handing them over
directly saves splits — which matters a lot when you only have 452 rows.

In [ ]:
def engineer(df):
    d = df.copy()
    if {"SystolicBP", "DiastolicBP"}.issubset(d.columns):
        d["PulsePressure"] = d["SystolicBP"] - d["DiastolicBP"]
        d["MAP"] = d["DiastolicBP"] + (d["SystolicBP"] - d["DiastolicBP"]) / 3
    if {"SystolicBP", "HeartRate"}.issubset(d.columns):
        d["ShockIndex"] = d["HeartRate"] / d["SystolicBP"].replace(0, np.nan)
    if {"SpO2", "HeartRate"}.issubset(d.columns):
        d["OxygenStrain"] = d["HeartRate"] / d["SpO2"].replace(0, np.nan)
    if "FallDetected" in d.columns:
        d["FallFlag"] = (d["FallDetected"] == "Yes").astype(int)
    return d


maternal_f = engineer(maternal)
monitoring_f = engineer(monitoring)

FEATURES_MATERNAL = ["Age", "SystolicBP", "DiastolicBP", "BloodSugar", "BodyTemp",
                     "HeartRate", "PulsePressure", "MAP", "ShockIndex"]
FEATURES_MONITOR = ["HeartRate", "SpO2", "SystolicBP", "DiastolicBP", "BodyTemp",
                    "FallFlag", "PulsePressure", "MAP", "ShockIndex", "OxygenStrain"]

X_mat, y_mat = maternal_f[FEATURES_MATERNAL], maternal_f["RiskLevel"]
X_mon, y_mon = monitoring_f[FEATURES_MONITOR], monitoring_f["Condition"]
print(X_mat.shape, X_mon.shape)

## Step 6 — Define the three models (twice)

**This is the part that actually decides your result.** XGBoost and LightGBM ship
with defaults tuned for tens of thousands of rows. On 452 rows, LightGBM's default
`num_leaves=31` + `min_child_samples=20` means it often can't find a valid split at
all and returns near-constant predictions.

So each model gets two configs: stock defaults, and small-data settings. We run
both so you can see the gap rather than take my word for it.

Note XGBoost needs **numeric** labels for multiclass, hence the `LabelEncoder`.

In [ ]:
def make_models(small_data: bool):
    """small_data=True -> configs appropriate for a few hundred rows."""
    if small_data:
        return {
            "RandomForest": RandomForestClassifier(
                n_estimators=500, min_samples_leaf=2, max_features="sqrt",
                class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
            "XGBoost": XGBClassifier(
                n_estimators=300, learning_rate=0.03, max_depth=3,
                min_child_weight=3, subsample=0.8, colsample_bytree=0.8,
                reg_lambda=2.0, tree_method="hist", eval_metric="mlogloss",
                n_jobs=-1, random_state=RANDOM_STATE),
            "LightGBM": LGBMClassifier(
                n_estimators=300, learning_rate=0.03, num_leaves=6,
                min_child_samples=5, subsample=0.8, subsample_freq=1,
                colsample_bytree=0.8, reg_lambda=1.0, class_weight="balanced",
                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
        }
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            n_estimators=300, learning_rate=0.1, max_depth=6,
            tree_method="hist", eval_metric="mlogloss",
            n_jobs=-1, random_state=RANDOM_STATE),
        "LightGBM": LGBMClassifier(
            n_estimators=300, learning_rate=0.1, num_leaves=31,
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    }


def benchmark(X, y, small_data, extra=None, label=""):
    """5-fold CV + holdout for each model. Returns (results_df, fitted, split)."""
    le = LabelEncoder().fit(y)
    y_enc = pd.Series(le.transform(y), index=y.index)

    Xtr, Xte, ytr, yte = train_test_split(
        X, y_enc, test_size=0.2, stratify=y_enc, random_state=RANDOM_STATE)

    models = make_models(small_data)
    if extra:
        models.update(extra)

    rows, fitted = [], {}
    for name, model in models.items():
        t0 = time.time()
        cv_scores = cross_val_score(model, X, y_enc, cv=CV, scoring="f1_macro", n_jobs=-1)
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        try:
            auc = roc_auc_score(yte, model.predict_proba(Xte), multi_class="ovr", average="macro")
        except Exception:
            auc = np.nan
        rows.append({
            "model": name,
            "cv_f1_macro": round(cv_scores.mean(), 4),
            "cv_std": round(cv_scores.std(), 4),
            "test_acc": round(accuracy_score(yte, pred), 4),
            "test_f1": round(f1_score(yte, pred, average="macro"), 4),
            "auc_ovr": round(auc, 4) if not np.isnan(auc) else None,
            "fit_sec": round(time.time() - t0, 1),
        })
        fitted[name] = model

    res = pd.DataFrame(rows).sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)
    print(f"\n=== {label} ===")
    print(res.to_string(index=False))
    return res, fitted, (Xtr, Xte, ytr, yte, le)

## Step 7 — Maternal dataset: all three, defaults vs tuned

SVM-RBF is included as a baseline. On small low-dimensional tabular data it is
genuinely competitive with tree ensembles — worth knowing before you commit.

In [ ]:
svm_baseline = {"SVM_RBF (baseline)": Pipeline([
    ("sc", StandardScaler()),
    ("m", SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE))])}

res_def, _, _ = benchmark(X_mat, y_mat, small_data=False,
                          label="MATERNAL — stock defaults")

res_tuned, fitted_mat, split_mat = benchmark(X_mat, y_mat, small_data=True,
                                             extra=svm_baseline,
                                             label="MATERNAL — small-data tuned")

comparison = (res_def.set_index("model")[["cv_f1_macro"]]
              .rename(columns={"cv_f1_macro": "defaults"})
              .join(res_tuned.set_index("model")[["cv_f1_macro"]]
                    .rename(columns={"cv_f1_macro": "tuned"}), how="inner"))
comparison["gain"] = (comparison["tuned"] - comparison["defaults"]).round(4)
print("\nEffect of tuning for small data:")
print(comparison.to_string())

## Step 8 — What the duplicates were hiding

Same pipeline, duplicates left in. Compare the number to Step 7.

In [ ]:
maternal_dupes = clean_maternal(maternal_raw, drop_duplicates=False)
Xd = engineer(maternal_dupes)[FEATURES_MATERNAL]
yd = maternal_dupes["RiskLevel"]

res_leak, _, _ = benchmark(Xd, yd, small_data=True, label="MATERNAL — duplicates LEFT IN")

best_clean = res_tuned[res_tuned.model != "SVM_RBF (baseline)"]["cv_f1_macro"].max()
best_leak = res_leak["cv_f1_macro"].max()
print(f"\nBest F1 with duplicates : {best_leak:.4f}")
print(f"Best F1 deduplicated    : {best_clean:.4f}")
print(f"INFLATION               : {best_leak - best_clean:+.4f}  "
      f"<-- report the deduplicated number")

## Step 9 — Monitoring dataset (60k rows)

This is where boosting earns its reputation. Watch the `fit_sec` column, not just
the accuracy — that's the real difference between the three at this scale.

In [ ]:
res_mon, fitted_mon, split_mon = benchmark(X_mon, y_mon, small_data=False,
                                           label="MONITORING — 60k rows")

Xtr, Xte, ytr, yte, le_mon = split_mon
best_mon_name = res_mon.iloc[0]["model"]
best_mon = fitted_mon[best_mon_name]
pred = best_mon.predict(Xte)
print(f"\nBest: {best_mon_name}")
print(classification_report(yte, pred, target_names=le_mon.classes_, digits=3))

## Step 10 — Confusion matrices and feature importance

In [ ]:
def plot_confusion(model, Xte, yte, classes, title, ax):
    cm = confusion_matrix(yte, model.predict(Xte))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(classes)), classes, fontsize=8)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title, fontsize=10)


Xtr_m, Xte_m, ytr_m, yte_m, le_mat = split_mat
best_mat_name = res_tuned.iloc[0]["model"]
best_mat = fitted_mat[best_mat_name]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_confusion(best_mat, Xte_m, yte_m, le_mat.classes_,
               f"Maternal — {best_mat_name}", axes[0])
plot_confusion(best_mon, Xte, yte, le_mon.classes_,
               f"Monitoring — {best_mon_name}", axes[1])
plt.tight_layout(); plt.show()

# model comparison bar chart
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = res_tuned[res_tuned.model.isin(["RandomForest", "XGBoost", "LightGBM"])]
ax.bar(plot_df["model"], plot_df["cv_f1_macro"],
       yerr=plot_df["cv_std"], capsize=5, color="#4C72B0")
ax.set_ylabel("CV F1-macro"); ax.set_title("Maternal — the three ensembles (tuned)")
ax.set_ylim(0, 1)
for i, v in enumerate(plot_df["cv_f1_macro"]):
    ax.text(i, v + 0.03, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

# feature importance from the winners
for name, model, X_ref in [("Maternal", best_mat, X_mat), ("Monitoring", best_mon, X_mon)]:
    if hasattr(model, "feature_importances_"):
        imp = (pd.Series(model.feature_importances_, index=X_ref.columns)
               .sort_values(ascending=True))
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.barh(imp.index, imp.values, color="#55A868")
        ax.set_title(f"{name} — feature importance"); plt.tight_layout(); plt.show()

## Step 11 — What issues might each patient be facing?

Rule engine using published thresholds (ACOG/AHA for blood pressure, ADA for
glucose, WHO for SpO2, 38.0 °C for fever). It covers single-vital flags *and*
cross-vital syndrome patterns — fever + tachycardia reads as a SIRS/sepsis
pattern, hypotension + tachycardia as shock.

In [ ]:
SEVERITY = {"info": 0, "monitor": 1, "concerning": 2, "urgent": 3}


def assess(row, units="F", pregnant=True):
    out = []
    sbp, dbp = row.get("SystolicBP"), row.get("DiastolicBP")
    hr, temp = row.get("HeartRate"), row.get("BodyTemp")
    spo2, bs, age = row.get("SpO2"), row.get("BloodSugar"), row.get("Age")

    # --- blood pressure ---
    if pd.notna(sbp) and pd.notna(dbp):
        if sbp >= 160 or dbp >= 110:
            out.append(("Severe-range hypertension (>=160/110)", "urgent"))
            if pregnant:
                out.append(("Screen for pre-eclampsia / eclampsia", "urgent"))
        elif sbp >= 140 or dbp >= 90:
            out.append(("Stage 2 hypertension (>=140/90)", "concerning"))
            if pregnant:
                out.append(("Possible gestational hypertension", "concerning"))
        elif sbp >= 130 or dbp >= 80:
            out.append(("Stage 1 hypertension (130-139/80-89)", "monitor"))
        elif sbp >= 120:
            out.append(("Elevated blood pressure", "monitor"))
        if sbp < 90 or dbp < 60:
            out.append(("Hypotension (<90/60)", "concerning"))
        pp = sbp - dbp
        if pp < 25:
            out.append(("Narrow pulse pressure (<25 mmHg)", "monitor"))
        elif pp > 60:
            out.append(("Wide pulse pressure (>60 mmHg)", "monitor"))

    # --- heart rate ---
    if pd.notna(hr):
        if hr < 40:
            out.append(("HR reading implausible (<40) - check sensor", "info"))
        elif hr < 60:
            out.append(("Bradycardia (<60 bpm)", "monitor"))
        elif hr > 120:
            out.append(("Marked tachycardia (>120 bpm)", "urgent"))
        elif hr > 100:
            out.append(("Tachycardia (>100 bpm)", "concerning"))

    # --- glucose (mmol/L) ---
    if pd.notna(bs):
        if bs >= 11.1:
            out.append(("Hyperglycaemia, diabetic range (>=11.1 mmol/L)", "urgent"))
            out.append(("Possible gestational or pre-existing diabetes", "concerning"))
        elif bs >= 7.8:
            out.append(("Impaired glucose tolerance (7.8-11.0 mmol/L)", "concerning"))
        elif bs < 3.9:
            out.append(("Hypoglycaemia (<3.9 mmol/L)", "urgent"))

    # --- SpO2 ---
    if pd.notna(spo2):
        if spo2 < 88:
            out.append(("Severe hypoxaemia (<88%) - oxygen needed", "urgent"))
        elif spo2 < 90:
            out.append(("Significant hypoxaemia (88-89%)", "urgent"))
        elif spo2 < 95:
            out.append(("Mild hypoxaemia (90-94%)", "concerning"))

    # --- temperature ---
    fever = False
    if pd.notna(temp):
        if units == "F":
            if temp >= 102.2:
                out.append(("High fever (>=102.2 F)", "urgent")); fever = True
            elif temp >= 100.4:
                out.append(("Fever (>=100.4 F) - possible infection", "concerning")); fever = True
            elif temp >= 99.5:
                out.append(("Low-grade fever", "monitor"))
            elif temp < 96.8:
                out.append(("Hypothermia (<96.8 F)", "concerning"))
        else:
            if temp >= 39.0:
                out.append(("High fever (>=39.0 C)", "urgent")); fever = True
            elif temp >= 38.0:
                out.append(("Fever (>=38.0 C) - possible infection", "concerning")); fever = True
            elif temp >= 37.5:
                out.append(("Low-grade fever", "monitor"))
            elif temp < 36.0:
                out.append(("Hypothermia (<36.0 C)", "concerning"))

    # --- maternal age ---
    if pregnant and pd.notna(age):
        if age < 15:
            out.append(("Very young maternal age (<15)", "urgent"))
        elif age < 18:
            out.append(("Adolescent pregnancy (<18)", "concerning"))
        elif age >= 40:
            out.append(("Advanced maternal age (>=40)", "concerning"))
        elif age >= 35:
            out.append(("Advanced maternal age (35-39)", "monitor"))

    if row.get("FallDetected") in (True, "Yes", 1):
        out.append(("Fall detected - injury / syncope assessment", "urgent"))

    # --- cross-vital syndrome patterns ---
    if fever and pd.notna(hr) and hr > 100:
        out.append(("SIRS pattern (fever + tachycardia) - sepsis workup", "urgent"))
    if pd.notna(sbp) and pd.notna(hr) and sbp < 90 and hr > 100:
        out.append(("Shock pattern (hypotension + tachycardia)", "urgent"))
    if pd.notna(spo2) and pd.notna(hr) and spo2 < 92 and hr > 100:
        out.append(("Cardiorespiratory distress", "urgent"))
    if pd.notna(sbp) and pd.notna(bs) and sbp >= 140 and bs >= 11.1:
        out.append(("Metabolic cluster (hypertension + hyperglycaemia)", "concerning"))
    if pd.notna(sbp) and pd.notna(hr) and sbp >= 160 and hr > 100:
        out.append(("Hypertensive emergency pattern", "urgent"))

    seen, uniq = set(), []
    for t, s in out:
        if t not in seen:
            seen.add(t); uniq.append((t, s))

    if any(s == "urgent" for _, s in uniq):
        band = "URGENT - clinician review"
    elif any(s == "concerning" for _, s in uniq):
        band = "PRIORITY - same-day review"
    elif uniq:
        band = "ROUTINE - monitor"
    else:
        band = "STABLE - no flags"

    return {"issue_count": len(uniq),
            "severity_score": sum(SEVERITY[s] for _, s in uniq),
            "triage_band": band,
            "issues": "; ".join(t for t, _ in uniq) if uniq else "No abnormal findings"}


def assess_frame(df, units="F", pregnant=True):
    return pd.DataFrame([assess(r, units, pregnant) for _, r in df.iterrows()],
                        index=df.index)


mat_annotated = pd.concat([maternal, assess_frame(maternal, "F", True)], axis=1)
mat_annotated["predicted_risk"] = le_mat.inverse_transform(best_mat.predict(X_mat))

mon_sample = monitoring.sample(8000, random_state=RANDOM_STATE).reset_index(drop=True)
mon_annotated = pd.concat([mon_sample, assess_frame(mon_sample, "C", False)], axis=1)
mon_annotated["predicted_condition"] = le_mon.inverse_transform(
    best_mon.predict(engineer(mon_sample)[FEATURES_MONITOR]))

print("Maternal triage bands:")
print(mat_annotated["triage_band"].value_counts().to_string())
print("\nTriage band vs labelled risk:")
print(pd.crosstab(mat_annotated["triage_band"], mat_annotated["RiskLevel"]).to_string())
print("\nMost common maternal issues:")
print(mat_annotated["issues"].str.split("; ").explode().value_counts().head(10).to_string())

mat_annotated.head(10)

## Step 12 — Save and download everything

In [ ]:
joblib.dump({"model": best_mat, "features": FEATURES_MATERNAL,
             "label_encoder": le_mat, "name": best_mat_name},
            "model_maternal_risk.joblib", compress=3)
joblib.dump({"model": best_mon, "features": FEATURES_MONITOR,
             "label_encoder": le_mon, "name": best_mon_name},
            "model_condition.joblib", compress=3)

mat_annotated.to_csv("maternal_annotated.csv", index=False)
mon_annotated.to_csv("monitoring_annotated_sample.csv", index=False)

leaderboard = pd.concat([
    res_tuned.assign(dataset="maternal_tuned"),
    res_def.assign(dataset="maternal_defaults"),
    res_leak.assign(dataset="maternal_WITH_duplicates"),
    res_mon.assign(dataset="monitoring"),
])
leaderboard.to_csv("model_leaderboard.csv", index=False)
print(leaderboard.to_string(index=False))

for f in ["model_maternal_risk.joblib", "model_condition.joblib",
          "maternal_annotated.csv", "monitoring_annotated_sample.csv",
          "model_leaderboard.csv"]:
    files.download(f)

## Step 13 — Where to go next

**The mid-risk class is a feature problem, not a model problem.** It sits between
low and high on every vital you have, so no amount of algorithm-swapping fixes it.
Two things that actually would:

1. **Collapse to binary** (high vs not-high) — fastest meaningful gain.
   ```python
   y_binary = (y_mat == "high risk").map({True: "high", False: "not_high"})
   benchmark(X_mat, y_binary, small_data=True, label="MATERNAL — binary")
   ```

2. **Add the features you're missing.** BMI, gestational diabetes status and
   prior-pregnancy complications are what the literature says matter most, and
   your current data has none of them:
   - Kurigram/Bangladesh set — `https://www.sciencedirect.com/science/article/pii/S2352340925000952`
   - Mendeley — `https://data.mendeley.com/datasets/p5w98dvbbk/1`
   - Zenodo — DOI `10.5281/zenodo.14537882`

**Before merging any of them, hash-check.** Several Kaggle "maternal health"
uploads are re-uploads of the same 1014 UCI rows under different names, and
concatenating them rebuilds the exact duplicate-leakage problem Step 8 measures:

```python
import hashlib
print(hashlib.md5(open("newfile.csv", "rb").read()).hexdigest())
```

Also check units: your blood sugar is **mmol/L**; most US datasets use **mg/dL**
(divide by 18.0).

**For Kaggle downloads**, upload your `kaggle.json` token first:
```python
from google.colab import files; files.upload()      # pick kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d andrewmvd/fetal-health-classification --unzip
```

**Don't chase the 95% on the monitoring set.** `Healthy` classifies at perfect
1.000 precision *and* recall because the synthetic generator built it from
non-overlapping vital ranges. The model is recovering the generator's rules,
not learning medicine.